In [1]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import random
from pomelo.utils import Hal

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/psycopg2/__init__.py:144: UserWarning: The psycopg2 wheel package will be renamed from release 2.8; in order to keep installing from binary please use "pip install psycopg2-binary" instead. For details see: <http://initd.org/psycopg/docs/install.html#binary-install-from-pypi>.
  """)


In [2]:
end_date = date.today().strftime("%Y-%m-%d")
six_months = date.today() - relativedelta(months=+6)
start_date = six_months.strftime("%Y-%m-%d")
# use six month-> don't have spl = purpose

In [3]:
print(str(end_date))
print(str(start_date))

2021-10-23
2021-04-23


In [4]:
hal = Hal()

sql = f"""
select distinct pd_sold.year_week_sold
        ,pd_sold.year_month_id
        ,pd_sold.id_shop
        ,pd_sold.id_warehouse
        --,pd_sold.id_store
        ---- By week sold, month id, shop, warehouse, store, product launched  ----
        
        ---- total no. of style for exiting products  ----
        ,case when sum(exist_styles.active_product) is null then 0
            else sum(exist_styles.active_product) end as tot_product_existing
            
         ---- total no. of style for new arivals [1 weeks] ----    
        ,case when pd_launched.product_launched is null then 0 
            else pd_launched.product_launched end as tot_product_launched 
            
        -- total no of style
        ,tot_product_existing+tot_product_launched tot_products        
        
        ---- sales unit new products  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.new_pd_sold) <= 0 then 0
            else sum(pd_sold.new_pd_sold) 
            end as new_pd_sold
         
         ---- sales unit new products  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.old_pd_sold) <= 0 then 0
            else sum(pd_sold.old_pd_sold) 
            end as old_pd_sold
            
        ---- ** PROPORTION NEW PRODUCT **  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.new_pd_sold) <= 0 then 0
            else round(cast(sum(pd_sold.new_pd_sold) AS float)/cast(sum(pd_sold.net_units_sold) AS float),2) 
            end as prop_new_pd
        ---- ** PROPORTION OLD PRODUCT **  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.old_pd_sold) <= 0 then 0
            else round(cast(sum(pd_sold.old_pd_sold) AS float)/cast(sum(pd_sold.net_units_sold) AS float),2) 
            end as prop_existing_pd
            

        from
        (----/* PRODUCT SOLD FOR EXISTING STYLES */----
        select distinct t.year_week_sold
        ,t.year_month_id
        ,t.id_shop
        ,t.id_warehouse
        ,t.id_store
        --- new & old - normal & lookbook ---
        ,case when product_age = 'new' then sum(t.tot_net_units_sold) else 0 end as new_pd_sold
        ,case when product_age = 'old' then sum(t.tot_net_units_sold) else 0 end as old_pd_sold

        ,new_pd_sold + old_pd_sold as net_units_sold
        from 
        (--- fact sales ---
        select dp2.id_product 
        ,fso2.id_shop
        --- case erply_store_id ---
        ,case when fso2.id_store = '5b989110a9afac06f24b80ce' then '231'
            when fso2.id_store = '5b55a681868e636a8ab1eaa2' then '261'
            when fso2.id_store = '5bb2e6ac1e5abf2252df039c' then '251'
            when fso2.id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
            when fso2.id_store = '5b9890f4d49bab0743f5689e' then '241'
            when fso2.id_store = '5a7c011b6878b592402d76bd' then '371'
            else fso2.id_store end as id_store
        ,case when id_shop = '2' and year_week_id < '202017' then '1'
            when id_shop = '2' and year_week_id >= '202017' then '11'
            when id_shop = '11' and year_week_id < '202038' then '1'
            when id_shop = '11' and year_week_id >= '202038' then '11'
            when id_shop = '10' then '1'
            when id_shop = '12' then '1'
            when id_shop = '4' then '1'
            when id_shop = '14' then '11'
            else id_shop end as id_warehouse
        ,ca.year_week_id as year_week_sold
        ,ca.year_month_id
        --- first date available ---
        ,min(pd_avl.year_week_available) as year_week_avl
        
        --- product age ---
        
        ,case when year_week_avl = ca.year_week_id then 'new'
        --when ca.year_week_id - year_week_avl = 1 then 'new' 
        --when ca.year_week_id - year_week_avl = 2 then 'new' 
        else 'old' end as product_age
        ,coalesce (sum(CASE WHEN fso2.transaction_type ='Return' THEN -fso2.product_quantity ELSE fso2.product_quantity END),0) as tot_net_units_sold
        from dwh.fact_sales_offline fso2
        ---/* LOOKBOOK COLLECTION */---
        left join (select distinct id_product
        ,sku_complete
        ,date_released
        from dwh.dim_product
        where date_released between '{start_date}' and '{end_date}'
        and id_product is not null
        and original_price_th is not NULL 
        and date_released is not NULL 
        and parent_product_line is not null
        and product_name is not null
        and product_line is not null
        and original_price_th != 0
        and henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
        and parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
        and id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
        and active = 1
        ) dp2
        on fso2.id_product = dp2.id_product
        ----/* Available date from NS */----
        left join (select sku_complete
        ,to_location_name
        ,case when to_id = 14 then 11
            when to_id = 29 then 211
            when to_id = 31 then 221
            when to_id = 69 then 231
            when to_id = 70 then 241
            when to_id = 71 then 251
            when to_id = 72 then 261
            when to_id = 73 then 281
            when to_id = 81 then 12
            when to_id = 92 then 311
            when to_id = 93 then 341
            when to_id = 95 then 321
            when to_id = 96 then 291
            when to_id = 99 then 301
            when to_id = 102 then 361
            when to_id = 103 then 331
            when to_id = 115 then 351
            when to_id = 116 then 381
            when to_id = 130 then 15
            when to_id = 131 then 371
            when to_id = 132 then 32
            when to_id = 159 then 391
            when to_id = 163 then 35
            when to_id = 165 then 42
            when to_id = 168 then 401
            when to_id = 175 then 411
            when to_id = 177 then 431
            when to_id = 178 then 111
            when to_id = 182 then 391
            when to_id = 186 then 441
            else null
            end as id_store
        ,min(ca.year_week_id) as year_week_available
        ,min(date(date_received)) as first_available_date
        FROM dwh.fact_ns_transfer_order ns
        ----/* YEAR WEEK ID FOR FIRST AVAILABLE DATE  */----
        left join (select full_date
        , case when ca.week_of_year_number < 10 then concat(convert(varchar(4),ca.year_id),right('0'+cast(ca.week_of_year_number AS varchar(2)),2))
            when ca.week_of_year_number = 53 then concat(convert(varchar(4),(ca.year_id+1)),'01')
            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(ns.date_received) = ca.full_date 
        where id_store is not null
        GROUP BY 1,2,3
        order by sku_complete
        ) pd_avl
        on dp2.sku_complete = pd_avl.sku_complete and pd_avl.id_store = fso2.id_store 
        ----/* YEAR WEEK ID FOR TRANSACTION  */----
        left join (select full_date
        ,case when ca.month_id < 10
            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.month_id AS VARCHAR(2)),2))
            else concat(ca.year_id,ca.week_of_year_number) end as year_month_id
        , case when ca.week_of_year_number < 10
            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
            when ca.week_of_year_number = 53
            then concat(convert(varchar(4),(ca.year_id+1)),'01')
            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(fso2.transaction_time) = ca.full_date
        where fso2.revenue_usd >0
        ---- NOTE: Marketing spend started 2019 only ----
        --and transaction_time >= date('2019-01-01')
        and sales_id is not null
        and dp2.id_product is not null
        and pd_avl.year_week_available is not null
        group by dp2.id_product
            ,fso2.id_shop
            ,fso2.id_store
            ,ca.year_week_id
            ,ca.year_month_id
        having year_week_sold >= year_week_avl) t
        group by t.year_week_sold
            ,t.id_shop
            ,t.year_month_id
            ,t.id_warehouse
            ,t.id_store
            ,product_age) pd_sold
        ----/* EXISTING PRODUCT */----
        ---- get count total active product ----
        ---- group by shop, store, week, lookbook ----
        left join 
        (select ca.year_week_id
        ,case when fisom.id_shop is null then '1' else fisom.id_shop end as id_shop
        ,case when id_store = '5b989110a9afac06f24b80ce' then '231'
            when id_store = '5b55a681868e636a8ab1eaa2' then '261'
            when id_store = '5bb2e6ac1e5abf2252df039c' then '251'
            when id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
            when id_store = '5b9890f4d49bab0743f5689e' then '241'
            when id_store = '5a7c011b6878b592402d76bd' then '371'
            else id_store
            end as id_store
        ,count(distinct dp2.id_product) as active_product
        from dwh.fact_inventory_snapshot_offline_master fisom 
        left join (select distinct id_product
        from dwh.dim_product dp
        where dp.date_released between '{start_date}' and '{end_date}'
        and dp.id_product is not null
        and dp.original_price_th is not NULL 
        and dp.date_released is not NULL 
        and dp.parent_product_line is not null
        and dp.product_name is not null
        and dp.product_line is not null
        and dp.original_price_th != 0
        and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
        --,'Lingerie Bodysuit', 'Sportswear Bottoms', 'Sportswear Outerwear', 'Sportswear Tops', 'Swimwear')
        and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
        and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
        and dp.active = 1) dp2
        on fisom.id_product = dp2.id_product	
        left join (select full_date
        , case when ca.week_of_year_number < 10
        then concat(convert(varchar(4),ca.year_id),right('0'+cast(ca.week_of_year_number AS varchar(2)),2))
        when ca.week_of_year_number = 53
        then concat(convert(varchar(4),(ca.year_id+1)),'01')
        else concat(ca.year_id,ca.week_of_year_number) 
        end as year_week_id
        from dwh.dim_calendar ca) ca
        on fisom.snapshot_date = ca.full_date
        where active = 1
        and fisom.snapshot_date between date'{start_date}' and '{end_date}'
        and id_store not in ('1','201', '25', '271')	
        group by id_shop
            ,id_store
            ,ca.year_week_id) exist_styles
        on pd_sold.year_week_sold = exist_styles.year_week_id and pd_sold.id_store = exist_styles.id_store
        ----/* NEW PRODUCT LAUNCHED */----
        ---- get total product launched in that week ----
        left join (select distinct year_week_available
        ,id_store
        ,sum(tot_product_launched) as product_launched 
        from (select distinct avl_store.year_week_available
        ,avl_store.id_store
        ,count(distinct avl_store.id_product) as tot_product_launched
        from (select dp2.id_product
        ,dp2.sku_complete 
        ,to_location_name
        ,case when to_id = 14 then 11
            when to_id = 29 then 211
            when to_id = 31 then 221
            when to_id = 69 then 231
            when to_id = 70 then 241
            when to_id = 71 then 251
            when to_id = 72 then 261
            when to_id = 73 then 281
            when to_id = 81 then 12
            when to_id = 92 then 311
            when to_id = 93 then 341
            when to_id = 95 then 321
            when to_id = 96 then 291
            when to_id = 99 then 301
            when to_id = 102 then 361
            when to_id = 103 then 331
            when to_id = 115 then 351
            when to_id = 116 then 381
            when to_id = 130 then 15
            when to_id = 131 then 371
            when to_id = 132 then 32
            when to_id = 159 then 391
            when to_id = 163 then 35
            when to_id = 165 then 42
            when to_id = 168 then 401
            when to_id = 175 then 411
            when to_id = 177 then 431
            when to_id = 178 then 111
            when to_id = 182 then 391
            when to_id = 186 then 441
            else null
            end as id_store
        ,min(ca.year_week_id) as year_week_available
        ,min(date(date_received)) as first_available_date
        from dwh.fact_ns_transfer_order ns
        left join (select full_date
        ,case when ca.week_of_year_number < 10
            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
            when ca.week_of_year_number = 53
            then concat(convert(varchar(4),(ca.year_id+1)),'01')
            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(ns.date_received) = ca.full_date 
        left join (select id_product
        ,sku_complete 
        from dwh.dim_product dp
        where dp.date_released between '{start_date}' and '{end_date}'
            and dp.id_product is not null
            and dp.original_price_th is not NULL 
            and dp.date_released is not NULL 
            and dp.parent_product_line is not null
            and dp.product_name is not null
            and dp.product_line is not null
            and dp.original_price_th != 0
            and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
            and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
            and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
            and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
            and dp.active = 1) dp2
        on ns.sku_complete = dp2.sku_complete 
        where id_store is not null
            and id_product is not null
        GROUP BY 1,2,3,4) avl_store
        where year_week_available is not null 
        and id_store is not null
        group by avl_store.year_week_available
            ,avl_store.id_store)
        group by 1,2) pd_launched
        on pd_sold.year_week_sold = pd_launched.year_week_available and pd_sold.id_store = pd_launched.id_store
        where active_product is not null
        group by pd_sold.year_week_sold
            ,pd_sold.year_month_id
            ,pd_sold.id_shop
            ,pd_sold.id_warehouse
            --,pd_sold.id_store
            ,pd_launched.product_launched
        order by pd_sold.year_week_sold

"""

Insert your Github token to access Vault:


GithubToken ········································


In [5]:
sale_prop = hal.get_pandas_df(sql)

In [6]:
sale_prop["tot_product_launched"] = sale_prop["tot_product_launched"].astype(float)
sale_prop["tot_product_existing"] = sale_prop["tot_product_existing"].astype(float)
sale_prop["tot_products"] = sale_prop["tot_products"].astype(float)

In [8]:
sale_prop["prop_new_styles"] = np.round(
    sale_prop["tot_product_launched"] / sale_prop["tot_products"], 2
)
sale_prop["prop_existing_styles"] = np.round(
    sale_prop["tot_product_existing"] / sale_prop["tot_products"], 2
)

In [9]:
sale_prop

,year_week_sold,year_month_id,id_shop,id_warehouse,tot_product_existing,tot_product_launched,tot_products,new_pd_sold,old_pd_sold,prop_new_pd,prop_existing_pd,prop_new_styles,prop_existing_styles
0,202117,202104,1,1,0.0,2.0,2.0,6,26,0.19,0.81,1.00,0.00
1,202117,202104,2,11,0.0,0.0,0.0,0,1064,0.00,1.00,NaN,NaN
2,202117,202104,1,1,0.0,6.0,6.0,302,0,1.00,0.00,1.00,0.00
3,202117,202104,2,11,0.0,3.0,3.0,30,592,0.05,0.95,1.00,0.00
4,202117,202104,5,5,0.0,24.0,24.0,612,604,0.50,0.50,1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,202143,202143,2,11,556.0,20.0,576.0,90,937,0.09,0.91,0.03,0.97
348,202143,202143,5,5,854.0,5.0,859.0,8,1354,0.01,0.99,0.01,0.99
349,202143,202143,1,1,672.0,12.0,684.0,153,1786,0.08,0.92,0.02,0.98
350,202143,202143,2,11,988.0,27.0,1015.0,181,2389,0.07,0.93,0.03,0.97


In [ ]:
sale_prop[sale_prop["year_week_id"] == "202114"]

In [10]:
sale_prop.rename(
    columns={
        "prop_new_pd": "prop_new_sold",
        "prop_existing_pd": "prop_existing_sold",
        "year_week_sold": "year_week_id",
    },
    inplace=True,
)

In [11]:
sale_prop

,year_week_id,year_month_id,id_shop,id_warehouse,tot_product_existing,tot_product_launched,tot_products,new_pd_sold,old_pd_sold,prop_new_sold,prop_existing_sold,prop_new_styles,prop_existing_styles
0,202116,202104,5,5,0.0,0.0,0.0,0,911,0.00,1.00,NaN,NaN
1,202116,202104,2,11,0.0,18.0,18.0,189,880,0.18,0.82,1.00,0.00
2,202116,202104,2,11,0.0,25.0,25.0,212,523,0.29,0.71,1.00,0.00
3,202117,202104,2,11,0.0,0.0,0.0,0,1064,0.00,1.00,NaN,NaN
4,202117,202104,2,11,36.0,18.0,54.0,148,598,0.20,0.80,0.33,0.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,202142,202142,2,11,1405.0,6.0,1411.0,6,6786,0.00,1.00,0.00,1.00
347,202142,202142,1,1,628.0,33.0,661.0,18,1055,0.02,0.98,0.05,0.95
348,202142,202142,2,11,730.0,10.0,740.0,12,1408,0.01,0.99,0.01,0.99
349,202142,202142,1,1,2284.0,31.0,2315.0,380,7461,0.05,0.95,0.01,0.99


In [11]:
sale_prop = sale_prop[
    [
        "year_week_id",
        "id_shop",
        "prop_new_sold",
        "prop_existing_sold",
        "tot_product_launched",
        "tot_product_existing",
    ]
].drop_duplicates()

In [12]:
sale_prop["id_shop"] = sale_prop["id_shop"].astype(str)

In [13]:
sale_prop  # there are duplicate becuase the data actually at id_store

,year_week_id,id_shop,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing
0,202115,2,0.59,0.41,23.0,0.0
1,202115,1,0.29,0.71,17.0,28.0
2,202115,1,1.00,0.00,14.0,196.0
3,202115,5,0.00,1.00,0.0,0.0
4,202116,1,0.00,1.00,17.0,26.0
...,...,...,...,...,...,...
346,202141,1,0.16,0.84,49.0,1156.0
347,202141,1,0.15,0.85,40.0,478.0
348,202141,1,0.11,0.89,35.0,1334.0
349,202141,5,0.01,0.99,5.0,698.0


In [14]:
sale_prop2 = (
    sale_prop.groupby(["year_week_id", "id_shop"]).mean().reset_index()
)  # so need to groupby first to make it at id_shop /week level

In [15]:
sale_prop2

,year_week_id,id_shop,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing
0,202115,1,0.645000,0.355000,15.500000,112.000000
1,202115,2,0.590000,0.410000,23.000000,0.000000
2,202115,5,0.000000,1.000000,0.000000,0.000000
3,202116,1,0.285000,0.715000,14.500000,377.000000
4,202116,2,0.360000,0.640000,27.000000,22.000000
...,...,...,...,...,...,...
84,202140,5,0.000000,1.000000,35.500000,273.000000
85,202141,1,0.146364,0.853636,47.272727,1086.545455
86,202141,11,0.140000,0.860000,24.000000,408.000000
87,202141,2,0.210000,0.790000,20.333333,812.666667


In [16]:
sale_prop2["new_sale_prop"] = (
    sale_prop2["prop_new_sold"] / sale_prop2["tot_product_launched"]
)
sale_prop2["new_existing_prop"] = (
    sale_prop2["prop_existing_sold"] / sale_prop2["tot_product_existing"]
)

In [17]:
sale_prop2

,year_week_id,id_shop,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing,new_sale_prop,new_existing_prop
0,202115,1,0.645000,0.355000,15.500000,112.000000,0.041613,0.003170
1,202115,2,0.590000,0.410000,23.000000,0.000000,0.025652,inf
2,202115,5,0.000000,1.000000,0.000000,0.000000,NaN,inf
3,202116,1,0.285000,0.715000,14.500000,377.000000,0.019655,0.001897
4,202116,2,0.360000,0.640000,27.000000,22.000000,0.013333,0.029091
...,...,...,...,...,...,...,...,...
84,202140,5,0.000000,1.000000,35.500000,273.000000,0.000000,0.003663
85,202141,1,0.146364,0.853636,47.272727,1086.545455,0.003096,0.000786
86,202141,11,0.140000,0.860000,24.000000,408.000000,0.005833,0.002108
87,202141,2,0.210000,0.790000,20.333333,812.666667,0.010328,0.000972


In [18]:
sale_prop_group = sale_prop2.groupby("id_shop")[
    [
        "prop_new_sold",
        "prop_existing_sold",
        "tot_product_launched",
        "tot_product_existing",
    ]
].mean()

In [19]:
sale_prop_group["new_sale_prop"] = (
    sale_prop_group["prop_new_sold"] / sale_prop_group["tot_product_launched"]
)
sale_prop_group["new_existing_prop"] = (
    sale_prop_group["prop_existing_sold"] / sale_prop_group["tot_product_existing"]
)

In [20]:
sale_prop_group

,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing,new_sale_prop,new_existing_prop
id_shop,,,,,,
1,0.179063,0.820937,20.492208,664.776469,0.008738,0.001235
11,0.060769,0.900769,10.461538,150.461538,0.005809,0.005987
2,0.130019,0.869981,19.935802,407.987037,0.006522,0.002132
5,0.087273,0.912727,18.681818,262.806818,0.004672,0.003473


In [21]:
hal = Hal()
sql = """
        select distinct main.id_store
              ,sum(main.prop_new_pd)/count(distinct main.year_week_sold) as avg_prop_new_pd
              ,sum(main.prop_existing_pd)/count(distinct main.year_week_sold) as avg_prop_existing_pd
        from (
            select distinct pd_sold.year_week_sold
                              ,pd_sold.id_store
                              ,case when sum(exist_styles.active_product) is null then 0
                                else sum(exist_styles.active_product) end as active_product
                              ,case when pd_launched.product_launched is null then 0 
                                else pd_launched.product_launched
                                end as tot_product_launched	   
                              ,case when sum(pd_sold.net_units_sold) = 0 then 0
                                else round(CAST(sum(pd_sold.new_pd_sold) AS float)/CAST(sum(pd_sold.net_units_sold) AS float),2) 
                                end as prop_new_pd
                              ,case when sum(pd_sold.net_units_sold) = 0 then 0
                                else round(CAST(sum(pd_sold.old_pd_sold) AS float)/CAST(sum(pd_sold.net_units_sold) AS float),2) 
                                end as prop_existing_pd
                        from (select distinct t.year_week_sold
                                        ,t.year_month_id
                                        ,t.id_shop
                                        ,t.id_warehouse
                                        ,t.id_store
                                        ,case when product_age = 'new' 
                                      then sum(t.tot_net_units_sold) 
                                      else 0
                                      end as new_pd_sold
                                      ,case when product_age = 'old' 
                                      then sum(t.tot_net_units_sold) 
                                      else 0
                                      end as old_pd_sold
                                      ,new_pd_sold + old_pd_sold as net_units_sold
                                from (	
                                        select dp2.id_product 
                                              ,fso2.id_shop
                                              ,case when fso2.id_store = '5b989110a9afac06f24b80ce' then '231'
                                                   when fso2.id_store = '5b55a681868e636a8ab1eaa2' then '261'
                                                   when fso2.id_store = '5bb2e6ac1e5abf2252df039c' then '251'
                                                   when fso2.id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
                                                   when fso2.id_store = '5b9890f4d49bab0743f5689e' then '241'
                                                   when fso2.id_store = '5a7c011b6878b592402d76bd' then '371'
                                                   when fso2.id_store = '39-1' then '391'
                                                   else fso2.id_store
                                                   end as id_store
                                              ,case when id_shop = '2' and year_week_id < '202017' then '1'
                                                    when id_shop = '2' and year_week_id >= '202017' then '11'
                                                    when id_shop = '11' and year_week_id < '202038' then '1'
                                                    when id_shop = '11' and year_week_id >= '202038' then '11'
                                                    WHEN id_shop = '10' THEN '1'
                                                    WHEN id_shop = '12' THEN '1'
                                                    WHEN id_shop = '4' THEN '1'
                                                     WHEN id_shop = '14' THEN '11'
                                                    else id_shop
                                                    end as id_warehouse
                                              ,ca.year_week_id as year_week_sold
                                              ,ca.year_month_id
                                              ,min(pd_avl.year_week_available) as year_week_avl
                                              ,case when year_week_avl = ca.year_week_id 
                                                    then 'new' 
                                                    else 'old'
                                                    end as product_age
                                              ,COALESCE(sum(CASE WHEN fso2.transaction_type ='Return' THEN -fso2.product_quantity ELSE fso2.product_quantity END),0) as tot_net_units_sold
                                        from dwh.fact_sales_offline fso2
                                        left join (select distinct id_product
                                                            ,sku_complete
                                                            ,date_released
                                                    from dwh.dim_product
                                                    where date_released between date('2016-12-26') and date(getdate()-1)
                                                    and id_product is not null
                                                    and original_price_th is not NULL 
                                                    and date_released is not NULL 
                                                    and parent_product_line is not null
                                                    and product_name is not null
                                                    and product_line is not null
                                                    and original_price_th != 0
                                                    and henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
                                                    and parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
                                                    and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
                                                    and id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
                                                    and active = 1
                                                    ) dp2
                                        on fso2.id_product = dp2.id_product
                                        left join (select sku_complete
                                                         ,to_location_name
                                                         ,case when to_id = 14 then 11
                                                               when to_id = 29 then 211
                                                               when to_id = 31 then 221
                                                               when to_id = 69 then 231
                                                               when to_id = 70 then 241
                                                               when to_id = 71 then 251
                                                               when to_id = 72 then 261
                                                               when to_id = 73 then 281
                                                               when to_id = 81 then 12
                                                               when to_id = 92 then 311
                                                               when to_id = 93 then 341
                                                               when to_id = 95 then 321
                                                               when to_id = 96 then 291
                                                               when to_id = 99 then 301
                                                               when to_id = 102 then 361
                                                               when to_id = 103 then 331
                                                               when to_id = 115 then 351
                                                               when to_id = 116 then 381
                                                               when to_id = 130 then 15
                                                               when to_id = 131 then 371
                                                               when to_id = 132 then 32
                                                               when to_id = 159 then 391
                                                               when to_id = 163 then 35
                                                               when to_id = 165 then 42
                                                               when to_id = 168 then 401
                                                               when to_id = 175 then 411
                                                               when to_id = 177 then 431
                                                               when to_id = 178 then 111
                                                               when to_id = 182 then 391
                                                               when to_id = 186 then 441
                                                          else null
                                                          end as id_store
                                                         ,min(ca.year_week_id) as year_week_available
                                                         ,min(date(date_received)) as first_available_date
                                                    FROM dwh.fact_ns_transfer_order ns
                                                    left join (select full_date
                                                                            , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                                            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                                            when ca.week_of_year_number = 53
                                                                            then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                                            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
                                                                     from dwh.dim_calendar ca) ca
                                                    on  date(ns.date_received) = ca.full_date 
                                                    where id_store is not null
                                                    GROUP BY 1,2,3
                                                    order by sku_complete
                                                    ) pd_avl
                                        on dp2.sku_complete = pd_avl.sku_complete and pd_avl.id_store = fso2.id_store 
                                        left join (select full_date
                                                            ,case when ca.month_id  in (1,2,3,4,5,6,7,8,9) 
                                           then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.month_id AS VARCHAR(2)),2))
                                        else concat(ca.year_id,ca.week_of_year_number) end as year_month_id
                                                                , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                                then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                                when ca.week_of_year_number = 53
                                                                then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                                else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
                                                            from dwh.dim_calendar ca) ca
                                        on  date(fso2.transaction_time) = ca.full_date
                                        where fso2.revenue_usd >0
                                        and transaction_time between date(getdate()-180) and date(getdate()-1)
                                        and sales_id is not null
                                        and dp2.id_product is not null
                                        and pd_avl.year_week_available is not null
                                        group by dp2.id_product
                                                ,fso2.id_shop
                                                ,fso2.id_store
                                                ,ca.year_week_id
                                                ,ca.year_month_id
                                        having year_week_sold >= year_week_avl
                                        ) t
                                group by t.year_week_sold
                                      ,t.id_shop
                                      ,t.year_month_id
                                      ,t.id_warehouse
                                      ,t.id_store
                                      ,product_age
                                    )pd_sold
                        left join (select ca.year_week_id
                                        ,case when fisom.id_shop is null then '1'
                                        else fisom.id_shop 
                                        end as id_shop
                                        ,case when id_store = '5b989110a9afac06f24b80ce' then '231'
                                           when id_store = '5b55a681868e636a8ab1eaa2' then '261'
                                           when id_store = '5bb2e6ac1e5abf2252df039c' then '251'
                                           when id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
                                           when id_store = '5b9890f4d49bab0743f5689e' then '241'
                                           when id_store = '5a7c011b6878b592402d76bd' then '371'
                                           else id_store
                                           end as id_store
                                        ,count(distinct dp2.id_product) as active_product
                                from dwh.fact_inventory_snapshot_offline_master fisom 
                                left join (select distinct id_product
                                                        from dwh.dim_product dp
                                                        where dp.date_released between date('2016-12-26') and date(getdate()-1)
                                                        and dp.id_product is not null
                                                        and dp.original_price_th is not NULL 
                                                        and dp.date_released is not NULL 
                                                        and dp.parent_product_line is not null
                                                        and dp.product_name is not null
                                                        and dp.product_line is not null
                                                        and dp.original_price_th != 0
                                                        and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
                                                        and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
                                                        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
                                                        and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
                                                        and dp.active = 1) dp2
                                on fisom.id_product = dp2.id_product	
                                left join (select full_date
                                                        , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                          then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                          when ca.week_of_year_number = 53
                                                          then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                          else concat(ca.year_id,ca.week_of_year_number) 
                                                          end as year_week_id
                                                    from dwh.dim_calendar ca) ca
                                on fisom.snapshot_date = ca.full_date
                                where active = 1
                                and fisom.snapshot_date between date('2018-06-22') and date(getdate()-1)
                                and id_store not in ('1','201', '25', '271')	
                                group by id_shop
                                        ,id_store
                                        , ca.year_week_id
                        ) exist_styles
                        on pd_sold.year_week_sold = exist_styles.year_week_id and pd_sold.id_store = exist_styles.id_store
                        left join (select distinct year_week_available
                                          ,id_store
                                          ,sum(tot_product_launched) as product_launched 
                                    from (select distinct avl_store.year_week_available
                                              ,avl_store.id_store
                                              ,count(distinct avl_store.id_product) as tot_product_launched
                                        from (select dp2.id_product
                                                   ,dp2.sku_complete 
                                                   ,to_location_name
                                                   ,case when to_id = 14 then 11
                                                       when to_id = 29 then 211
                                                       when to_id = 31 then 221
                                                       when to_id = 69 then 231
                                                       when to_id = 70 then 241
                                                       when to_id = 71 then 251
                                                       when to_id = 72 then 261
                                                       when to_id = 73 then 281
                                                       when to_id = 81 then 12
                                                       when to_id = 92 then 311
                                                       when to_id = 93 then 341
                                                       when to_id = 95 then 321
                                                       when to_id = 96 then 291
                                                       when to_id = 99 then 301
                                                       when to_id = 102 then 361
                                                       when to_id = 103 then 331
                                                       when to_id = 115 then 351
                                                       when to_id = 116 then 381
                                                       when to_id = 130 then 15
                                                       when to_id = 131 then 371
                                                       when to_id = 132 then 32
                                                       when to_id = 159 then 391
                                                       when to_id = 163 then 35
                                                       when to_id = 165 then 42
                                                       when to_id = 168 then 401
                                                       when to_id = 175 then 411
                                                       when to_id = 177 then 431
                                                       when to_id = 178 then 111
                                                       when to_id = 182 then 391
                                                       when to_id = 186 then 441
                                                              else null
                                                              end as id_store
                                                    ,min(ca.year_week_id) as year_week_available
                                                    ,min(date(date_received)) as first_available_date
                                                FROM dwh.fact_ns_transfer_order ns
                                                left join (select full_date
                                                                , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                                then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                                when ca.week_of_year_number = 53
                                                                then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                                else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
                                                            from dwh.dim_calendar ca) ca
                                                on  date(ns.date_received) = ca.full_date 
                                                left join (select id_product
                                                                 ,sku_complete 
                                                           from dwh.dim_product dp
                                                           where dp.date_released between date('2016-12-26') and date(getdate()-1)
                                                                and dp.id_product is not null
                                                                and dp.original_price_th is not NULL 
                                                                and dp.date_released is not NULL 
                                                                and dp.parent_product_line is not null
                                                                and dp.product_name is not null
                                                                and dp.product_line is not null
                                                                and dp.original_price_th != 0
                                                                and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
                                                                -- ,'Lingerie Bodysuit', 'Sportswear Bottoms', 'Sportswear Outerwear', 'Sportswear Tops', 'Swimwear')
                                                                and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
                                                                and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
                                                                and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
                                                                and dp.active = 1
                                                          ) dp2
                                                on ns.sku_complete = dp2.sku_complete 
                                                where id_store is not null
                                                and id_product is not null
                                                GROUP BY 1,2,3,4
                                                ) avl_store
                                        where year_week_available is not null 
                                        and id_store is not null
                                        group by avl_store.year_week_available
                                              ,avl_store.id_store)
                                    group by 1,2
                                ) pd_launched
                        on pd_sold.year_week_sold = pd_launched.year_week_available and pd_sold.id_store = pd_launched.id_store
                        where active_product is not null
                        group by pd_sold.year_week_sold
                                ,pd_sold.id_store
                                ,pd_launched.product_launched
        ) main
        group by main.id_store
        order by main.id_store
        """
store_dist_6mths = hal.get_pandas_df(sql)

In [22]:
store_dist_6mths

,id_store,avg_prop_new_pd,avg_prop_existing_pd
0,11,0.128500,0.871500
1,111,0.050000,0.950000
2,12,0.060741,0.939259
3,15,0.042727,0.957273
4,211,0.091500,0.908500
5,221,0.187500,0.812500
6,231,0.000000,1.000000
7,251,0.117000,0.883000
8,261,0.153636,0.846364
9,281,0.211500,0.788500


In [6]:
# if import error run this cell first
!pip install --upgrade google-api-python-client oauth2client
!pip install gspread

In [7]:
import boto3

s3 = boto3.resource("s3")

In [8]:
#### authenticate your identity using the JSON credentials and provide access to your Google Drive #####
from apiclient import discovery
from httplib2 import Http
import oauth2client
from oauth2client import file, client, tools

obj = lambda: None
lmao = {
    "auth_host_name": "localhost",
    "noauth_local_webserver": "store_true",
    "auth_host_port": [8080, 8090],
    "logging_level": "ERROR",
}
for k, v in lmao.items():
    setattr(obj, k, v)

# authorization boilerplate code
SCOPES = "https://www.googleapis.com/auth/drive.readonly"
store = file.Storage("token.json")
creds = store.get()
# creds = None
# The following will give you a link if token.json does not exist, the link allows the user to give this app permission
if not creds or creds.invalid:
    flow = client.flow_from_clientsecrets(
        "/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/online_dfm_v2/client_secret.json",
        SCOPES,
    )
    creds = tools.run_flow(flow, store, obj)

In [9]:
import io
from googleapiclient.http import MediaIoBaseDownload

DRIVE = discovery.build("drive", "v3", http=creds.authorize(Http()))
# https://docs.google.com/spreadsheets/d/1LKibyPJvNLSvPR4A5t3zcdEBPAxrFuy4/edit#gid=1796573573
file_id = "1LKibyPJvNLSvPR4A5t3zcdEBPAxrFuy4"
request = DRIVE.files().get_media(fileId=file_id)
# replace the filename and extension in the first field below
fh = io.FileIO("store_traffic.xlsx", mode="w")
downloader = MediaIoBaseDownload(fh, request)
done = False
while done is False:
    status, done = downloader.next_chunk()
    print("Download %d%%." % int(status.progress() * 100))

Download 100%.


In [10]:
df = pd.read_excel(
    "store_traffic.xlsx", engine="openpyxl", sheet_name="unpivot_forecast"
)

In [19]:
store_map = df[["Store_name", "id_store", "oldcluster", "country"]].drop_duplicates()

In [23]:
store_map = store_map[~store_map["id_store"].isna()]

In [25]:
store_map["id_store"] = store_map["id_store"].astype(int)
store_map["id_store"] = store_map["id_store"].astype(str)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':
/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  from ipykernel import kernelapp as app


In [26]:
store_map

,Store_name,id_store,oldcluster,country
0,Central World,261,sc2,TH
7,Icon Siam,11,sc2,TH
14,EmQuartier,251,sc2,TH
21,Mega Bangna,211,sc1,TH
28,Central Pinklao,221,sc3,TH
35,SG 313 Somerset,12,sc1,SG
42,Central Phuket,301,sc3,TH
49,Central Rama 9,311,sc3,TH
56,Central Rama 3,321,sc3,TH
63,Zpell,331,sc3,TH


In [27]:
store_dist_6mths

,id_store,avg_prop_new_pd,avg_prop_existing_pd
0,11,0.100500,0.899500
1,111,0.032000,0.968000
2,12,0.054444,0.945556
3,15,0.048636,0.951364
4,211,0.072000,0.928000
5,221,0.167500,0.832500
6,231,0.000000,1.000000
7,251,0.090000,0.910000
8,261,0.130909,0.869091
9,281,0.196500,0.803500


In [30]:
store_dist_6mths = store_dist_6mths.merge(
    store_map[["id_store", "oldcluster", "country"]], how="left", on="id_store"
)

In [34]:
store_dist_6mths = store_dist_6mths[~store_dist_6mths["country"].isna()]

In [35]:
store_dist_6mths

,id_store,avg_prop_new_pd,avg_prop_existing_pd,oldcluster,country
0,11,0.100500,0.899500,sc2,TH
1,111,0.032000,0.968000,sc2,MY
2,12,0.054444,0.945556,sc1,SG
3,15,0.048636,0.951364,sc3,ID
4,211,0.072000,0.928000,sc1,TH
5,221,0.167500,0.832500,sc3,TH
7,251,0.090000,0.910000,sc2,TH
8,261,0.130909,0.869091,sc2,TH
11,301,0.033704,0.966296,sc3,TH
12,311,0.116000,0.884000,sc3,TH


In [36]:
store_dist_6mths.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/offline_clothing_v2/data/deploy_store_dist_offline.csv",
    index=False,
)